# EDA full analysis for KT
Análise exploratória forte focada em sequência, tempo, sessões e adequação para KT.


In [ ]:
from pathlib import Path
import json, sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
IN_COLAB='google.colab' in sys.modules
REPO_URL='https://github.com/GuilhermeDesoler/ai-core.git'
REPO_DIR=Path('/content/ai-core')
if IN_COLAB:
    if not REPO_DIR.exists():
        get_ipython().system(f'git clone -b improve/high-impact-training {REPO_URL} /content/ai-core')
    get_ipython().run_line_magic('cd','/content/ai-core')
    get_ipython().system('pip install -q -r requirements.txt')
else:
    REPO_DIR=Path.cwd()
answers=pd.read_csv(REPO_DIR/'data/processed/dataset/answers_prepared.csv')
with open(REPO_DIR/'data/processed/sequences/user_sequences.json') as f:
    sequences=json.load(f)
answers['timestamp_dt']=pd.to_datetime(answers['timestamp'], unit='ms', errors='coerce')


In [ ]:
overview=pd.Series({'rows':len(answers),'users':answers['user_id'].nunique(),'questions':answers['question_id'].nunique(),'skills':answers['skill_id'].nunique(),'correct_rate':float(answers['correct'].mean()),'dup_rows':int(answers.duplicated().sum()),'time_zero_rate':float((answers['time_response']==0).mean())})
overview.to_frame('value')


In [ ]:
user_stats=answers.groupby('user_id').agg(n_interactions=('question_id','size'),n_skills=('skill_id','nunique'),first_ts=('timestamp_dt','min'),last_ts=('timestamp_dt','max')).reset_index()
user_stats['timespan_hours']=((user_stats['last_ts']-user_stats['first_ts']).dt.total_seconds()/3600).fillna(0)
display(user_stats[['n_interactions','n_skills','timespan_hours']].describe(percentiles=[0.1,0.25,0.5,0.75,0.9,0.95,0.99]))
fig=plt.figure(figsize=(10,5)); plt.hist(user_stats['n_interactions'], bins=50); plt.title('Interactions per user'); plt.show()


In [ ]:
answers=answers.sort_values(['user_id','timestamp']).copy()
answers['delta_t_ms']=answers.groupby('user_id')['timestamp'].diff().fillna(0)
answers['delta_t_min']=answers['delta_t_ms']/60000
display(pd.Series({'delta_p50_min':float(answers['delta_t_min'].quantile(0.5)),'delta_p90_min':float(answers['delta_t_min'].quantile(0.9)),'delta_p99_min':float(answers['delta_t_min'].quantile(0.99)),'time_p99_sec':float(answers['time_response'].quantile(0.99)/1000)}).to_frame('value'))
fig=plt.figure(figsize=(10,5)); plt.hist(np.log1p(answers['delta_t_ms']), bins=60); plt.title('log1p(delta_t_ms)'); plt.show()


In [ ]:
SESSION_GAP_MIN=60
answers['new_session_flag']=((answers['delta_t_min']>SESSION_GAP_MIN)|(answers.groupby('user_id').cumcount()==0)).astype(int)
answers['session_idx']=answers.groupby('user_id')['new_session_flag'].cumsum()
session_stats=answers.groupby(['user_id','session_idx']).agg(n_interactions=('question_id','size'),n_skills=('skill_id','nunique')).reset_index()
display(session_stats[['n_interactions','n_skills']].describe(percentiles=[0.1,0.25,0.5,0.75,0.9,0.95,0.99]))
fig=plt.figure(figsize=(10,5)); plt.hist(session_stats['n_interactions'], bins=40); plt.title('Session length'); plt.show()


In [ ]:
answers['prev_skill_id']=answers.groupby('user_id')['skill_id'].shift(1)
answers['skill_switch']=((answers['prev_skill_id'].notna())&(answers['skill_id']!=answers['prev_skill_id'])).astype(int)
frag=answers.groupby('user_id').agg(n_interactions=('question_id','size'),n_skills=('skill_id','nunique'),switches=('skill_switch','sum')).reset_index()
frag['switch_rate']=frag['switches']/(frag['n_interactions']-1).clip(lower=1)
frag['interactions_per_skill']=frag['n_interactions']/frag['n_skills'].clip(lower=1)
display(frag[['switch_rate','interactions_per_skill']].describe(percentiles=[0.1,0.25,0.5,0.75,0.9,0.95,0.99]))
fig=plt.figure(figsize=(10,5)); plt.hist(frag['switch_rate'], bins=40); plt.title('Skill switch rate'); plt.show()


In [ ]:
user_skill_counts=answers.groupby(['user_id','skill_id']).size().reset_index(name='n_attempts')
repeat_summary=pd.Series({'mean_attempts_per_user_skill':float(user_skill_counts['n_attempts'].mean()),'median_attempts_per_user_skill':float(user_skill_counts['n_attempts'].median()),'share_attempted_once':float((user_skill_counts['n_attempts']==1).mean()),'share_attempted_ge_3':float((user_skill_counts['n_attempts']>=3).mean())})
repeat_summary.to_frame('value')


In [ ]:
diagnostic=pd.DataFrame([{'criterion':'long user histories','value':float(user_stats['n_interactions'].median()),'comment':'higher is better for KT'},{'criterion':'session length','value':float(session_stats['n_interactions'].median()),'comment':'short sessions hurt temporal KT'},{'criterion':'skill switch rate','value':float(frag['switch_rate'].median()),'comment':'high values mean fragmented study'},{'criterion':'repeat per user-skill >=3','value':float((user_skill_counts['n_attempts']>=3).mean()),'comment':'higher helps mastery modeling'},{'criterion':'time_response zero rate','value':float((answers['time_response']==0).mean()),'comment':'high values reduce temporal signal'}])
diagnostic
